**Github** : https://github.com/OzgurYldrm/AI-ML-Course     
**Youtube** : https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence

Dataset paper: https://aclanthology.org/W03-0419/       

In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import gensim.downloader as api
from torch.nn.utils.rnn import pad_sequence
import torch
import torch.nn as nn
import torch.optim as optim

/home/ozgur/Desktop/Eğitim/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset

ID	Tag	Anlamı      
0	O	Entity değil        
1	B-PER	Person başlangıcı       
2	I-PER	Person devamı       
3	B-ORG	Organization başlangıcı     
4	I-ORG	Organization devamı     
5	B-LOC	Location başlangıcı     
6	I-LOC	Location devamı     
7	B-MISC	Miscellaneous başlangıcı        
8	I-MISC	Miscellaneous devamı        

In [4]:
dataset = load_dataset(
    "eriktks/conll2003",
    revision="convert/parquet"
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [6]:
print(type(dataset))
print(dataset["train"].column_names)

<class 'datasets.dataset_dict.DatasetDict'>
['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags']


In [5]:
dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [7]:
dataset = dataset.select_columns(["id", "tokens", "ner_tags"])

# Dataset-Dataloader

In [ ]:
glove = api.load("glove-wiki-gigaword-100")
words = dataset["train"][0]["tokens"]
vectors = []
for word in words:
    if word in glove:
        vectors.append(torch.tensor(glove[word]))
    else:
        vectors.append(torch.zeros(100))
vectors = torch.stack(vectors)
print(words)
print(vectors.shape)
print(vectors)

In [10]:
class NER_Dataset(Dataset):
    def __init__(self, dataset):
        self.glove = api.load("glove-wiki-gigaword-100")
        self.dataset = dataset

    def embed(self, words):
        vectors = []
        for word in words:
            if word in self.glove:
                vectors.append(torch.tensor(self.glove[word]))
            else:
                vectors.append(torch.zeros(100))
        return torch.stack(vectors)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        tokens = item["tokens"]
        labels = item["ner_tags"]
        embeddings = self.embed(tokens)
        labels = torch.tensor(labels)
        return embeddings, labels


In [ ]:
def ner_collate_fn(batch):
    embeddings, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in embeddings])
    padded_embeddings = pad_sequence(embeddings,batch_first=True)
    padded_labels = pad_sequence(labels,batch_first=True,padding_value=-100)
    return padded_embeddings, padded_labels, lengths

In [12]:
train_dataset = NER_Dataset(dataset["train"])
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,collate_fn=ner_collate_fn)

# Model

In [13]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
class NER_model(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=128, num_classes=9, dropout=0.3):
        super(NER_model, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):

        packed = pack_padded_sequence(
            x, 
            lengths.cpu(), 
            batch_first=True, 
            enforce_sorted=False
        )

        packed_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )
        lstm_out = self.dropout(lstm_out)
        logits = self.fc(lstm_out)
        return logits

In [14]:
model = NER_model(input_dim=100, hidden_dim=128,num_classes=9,dropout=0.3)
model.to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training

In [ ]:
for embeds,labels,lengths in train_loader:
    print(labels)
    break

torch.Size([32, 41])


In [19]:
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for embeddings, labels, lengths in train_loader:
        embeddings = embeddings.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(embeddings,lengths) # outputs: (B, T, C)
        B, T, C = outputs.shape
        outputs = outputs.view(B * T, C)
        labels = labels.view(B * T)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"epoch:{epoch} | ", total_loss / len(train_loader))

epoch:0 |  0.45756948184315327
epoch:1 |  0.2797202035818665
epoch:2 |  0.2387085198711154
epoch:3 |  0.21680158967309227
epoch:4 |  0.20046528634150643
epoch:5 |  0.18606526404619217
epoch:6 |  0.17635454298559244
epoch:7 |  0.167850051491027
epoch:8 |  0.15801651901427596
epoch:9 |  0.1500775898236077


# Test

Özel kelimeler ("EU" gibi) GloVe içerisinde bulunmadığı için hepsi 0 vektörü ile ifade ediliyor. NER gibi bir görev için rezalet bir durum. Ama amaç burda sadece mimarinin göreve nasıl uydurulabileceğini görmek o yüzden kasmayın 

In [20]:
test_dataset = NER_Dataset(dataset["test"])
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=ner_collate_fn
)
id2label = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC",
}

In [21]:
from sklearn.metrics import f1_score

def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for embeddings, labels, lengths in dataloader:
            embeddings = embeddings.to(device)
            labels = labels.to(device)
            outputs = model(embeddings,lengths)
            preds = torch.argmax(outputs, dim=-1)
            for i in range(len(lengths)):
                seq_len = lengths[i]
                gold = labels[i][:seq_len].cpu().tolist()
                pred = preds[i][:seq_len].cpu().tolist()
                all_labels.extend(gold)
                all_preds.extend(pred)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return f1

def show_examples(model, dataset, num_examples=5):
    model.eval()
    for i in range(num_examples):
        item = dataset.dataset[i]
        tokens = item["tokens"]
        gold_labels = item["ner_tags"]
        embeddings = dataset.embed(tokens).unsqueeze(0).to(device)
        lengths = torch.tensor([len(tokens)])
        with torch.no_grad():
            outputs = model(embeddings,lengths)
            preds = torch.argmax(outputs, dim=-1).squeeze(0)
        print(f"\nSentence {i+1}")
        print("-" * 40)
        for token, gold, pred in zip(tokens, gold_labels, preds.tolist()):
            print(
                f"{token:15} "
                f"Gold: {id2label[gold]:7} "
                f"Pred: {id2label[pred]:7}"
            )

In [22]:
f1 = evaluate(model, test_loader)
print("Test F1 Score:", f1)
show_examples(model, test_dataset, num_examples=5)

Test F1 Score: 0.6503363806912209

Sentence 1
----------------------------------------
SOCCER          Gold: O       Pred: O      
-               Gold: O       Pred: O      
JAPAN           Gold: B-LOC   Pred: O      
GET             Gold: O       Pred: O      
LUCKY           Gold: O       Pred: B-PER  
WIN             Gold: O       Pred: O      
,               Gold: O       Pred: O      
CHINA           Gold: B-PER   Pred: O      
IN              Gold: O       Pred: O      
SURPRISE        Gold: O       Pred: O      
DEFEAT          Gold: O       Pred: O      
.               Gold: O       Pred: O      

Sentence 2
----------------------------------------
Nadim           Gold: B-PER   Pred: B-LOC  
Ladki           Gold: I-PER   Pred: O      

Sentence 3
----------------------------------------
AL-AIN          Gold: B-LOC   Pred: O      
,               Gold: O       Pred: O      
United          Gold: B-LOC   Pred: B-LOC  
Arab            Gold: I-LOC   Pred: O      
Emirates       